# Ch.2 — Collaborative Filtering

> Find users with similar taste (user-based) or movies with similar rating patterns (item-based), and recommend accordingly. The first step toward personalisation.

**Dataset:** MovieLens 100k  
**Task:** Build user-based and item-based collaborative filtering  
**Outcome:** Item-based CF = ~68% HR@10

## The Core Idea

**Collaborative Filtering** makes predictions based on the collective behavior of all users:

- **User-Based CF**: Find users with similar rating patterns → recommend what they liked
- **Item-Based CF**: Find items with similar rating patterns → recommend items similar to what you liked

Both rely on **similarity metrics** (cosine, Pearson) applied to the sparse user-item matrix.

$$\hat{r}_{ui} = \bar{r}_u + \frac{\sum_{v \in \mathcal{N}_k(u)} \text{sim}(u, v) \cdot (r_{vi} - \bar{r}_v)}{\sum_{v \in \mathcal{N}_k(u)} |\text{sim}(u, v)|}$$

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

sns.set_theme(style="whitegrid", palette="muted")
SEED = 42
np.random.seed(SEED)

print("Libraries loaded.")

In [ ]:
# ── Load MovieLens 100k ───────────────────────────────────────────────────
url = "https://files.grouplens.org/datasets/movielens/ml-100k/"

ratings = pd.read_csv(
    url + "u.data", sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"]
)

n_users = ratings["user_id"].nunique()
n_items = ratings["item_id"].nunique()
print(f"Users: {n_users}  Items: {n_items}  Ratings: {len(ratings):,}")

In [ ]:
# ── Leave-One-Out Split ───────────────────────────────────────────────────
def leave_one_out_split(ratings_df):
    """Hold out the last rating per user (by timestamp) for testing."""
    ratings_sorted = ratings_df.sort_values("timestamp")
    test = ratings_sorted.groupby("user_id").tail(1).copy()
    train = ratings_sorted.drop(test.index).copy()
    return train, test

train, test = leave_one_out_split(ratings)
print(f"Train: {len(train):,}  Test: {len(test):,}")

In [ ]:
# ── Build Sparse User-Item Matrix ─────────────────────────────────────────
def build_sparse_matrix(train_df, n_users, n_items):
    """Create sparse CSR matrix from rating triplets (0-indexed)."""
    row = train_df["user_id"].values - 1
    col = train_df["item_id"].values - 1
    data = train_df["rating"].values.astype(np.float32)
    return csr_matrix((data, (row, col)), shape=(n_users, n_items))

R = build_sparse_matrix(train, n_users, n_items)
print(f"User-item matrix: {R.shape}")
print(f"Non-zero entries: {R.nnz:,}")
print(f"Sparsity: {1 - R.nnz / (R.shape[0] * R.shape[1]):.1%}")

In [ ]:
# ── Evaluation Metrics ────────────────────────────────────────────────────
def hit_rate_at_k(test_df, top_k_per_user, k=10):
    hits = 0
    for _, row in test_df.iterrows():
        user = row["user_id"]
        test_item = row["item_id"]
        recs = top_k_per_user.get(user, [])[:k]
        if test_item in recs:
            hits += 1
    return hits / len(test_df)

def ndcg_at_k(test_df, top_k_per_user, k=10):
    ndcgs = []
    for _, row in test_df.iterrows():
        user = row["user_id"]
        test_item = row["item_id"]
        recs = top_k_per_user.get(user, [])[:k]
        if test_item in recs:
            rank = recs.index(test_item) + 1
            ndcgs.append(1.0 / np.log2(rank + 1))
        else:
            ndcgs.append(0.0)
    return np.mean(ndcgs)

print("Evaluation functions defined.")

## User-Based Collaborative Filtering

Find the $k$ most similar users (by Pearson correlation on co-rated items) and predict ratings as a weighted average of their deviations from their mean:

$$\hat{r}_{ui} = \bar{r}_u + \frac{\sum_{v \in \mathcal{N}_k(u)} \text{sim}(u, v) \cdot (r_{vi} - \bar{r}_v)}{\sum_{v \in \mathcal{N}_k(u)} |\text{sim}(u, v)|}$$

In [ ]:
# ── User-Based CF ─────────────────────────────────────────────────────────
def user_based_cf(R_sparse, k_neighbors=50, n_recs=10):
    """User-based CF using cosine similarity on the sparse matrix."""
    R_dense = R_sparse.toarray()
    
    # Mean-center each user's ratings (only non-zero entries)
    user_means = np.zeros(R_dense.shape[0])
    R_centered = R_dense.copy()
    for u in range(R_dense.shape[0]):
        rated_mask = R_dense[u] > 0
        if rated_mask.sum() > 0:
            user_means[u] = R_dense[u, rated_mask].mean()
            R_centered[u, rated_mask] -= user_means[u]
            R_centered[u, ~rated_mask] = 0  # keep unrated as 0

    # User-user similarity (cosine on centered ratings)
    user_sim = cosine_similarity(csr_matrix(R_centered))
    np.fill_diagonal(user_sim, 0)
    
    recommendations = {}
    for u in range(R_dense.shape[0]):
        rated_mask = R_dense[u] > 0
        # Top-k similar users
        neighbor_sims = user_sim[u]
        top_neighbors = np.argsort(neighbor_sims)[-k_neighbors:]
        
        scores = np.zeros(R_dense.shape[1])
        for i in range(R_dense.shape[1]):
            if rated_mask[i]:
                continue  # skip already rated
            # Neighbors who rated item i
            neighbor_ratings = R_dense[top_neighbors, i]
            valid = neighbor_ratings > 0
            if valid.sum() == 0:
                continue
            sims = neighbor_sims[top_neighbors][valid]
            devs = neighbor_ratings[valid] - user_means[top_neighbors][valid]
            denom = np.abs(sims).sum()
            if denom > 0:
                scores[i] = user_means[u] + (sims * devs).sum() / denom
        
        top_items = np.argsort(scores)[-n_recs:][::-1]
        recommendations[u + 1] = (top_items + 1).tolist()
    
    return recommendations

print("Computing user-based CF (this may take a minute)...")
top_k_user_cf = user_based_cf(R, k_neighbors=50, n_recs=10)

hr_ucf = hit_rate_at_k(test, top_k_user_cf, k=10)
ndcg_ucf = ndcg_at_k(test, top_k_user_cf, k=10)
print(f"\nUser-Based CF:")
print(f"  HR@10   = {hr_ucf:.3f} ({hr_ucf*100:.1f}%)")
print(f"  NDCG@10 = {ndcg_ucf:.4f}")

## Item-Based Collaborative Filtering

Instead of finding similar users, find similar **items**. Item similarities are more stable (movies don't change genre) and can be precomputed.

$$\hat{r}_{ui} = \frac{\sum_{j \in \mathcal{N}_k(i)} \text{sim}(i, j) \cdot r_{uj}}{\sum_{j \in \mathcal{N}_k(i)} |\text{sim}(i, j)|}$$

In [ ]:
# ── Item-Based CF ─────────────────────────────────────────────────────────
def item_based_cf(R_sparse, k_neighbors=30, n_recs=10):
    """Item-based CF using cosine similarity."""
    # Item-item similarity (cosine on columns)
    item_sim = cosine_similarity(R_sparse.T)
    np.fill_diagonal(item_sim, 0)
    
    R_dense = R_sparse.toarray()
    recommendations = {}
    
    for u in range(R_dense.shape[0]):
        rated_items = np.where(R_dense[u] > 0)[0]
        user_ratings = R_dense[u]
        
        scores = np.zeros(R_dense.shape[1])
        for i in range(R_dense.shape[1]):
            if user_ratings[i] > 0:
                continue
            # Similarity to items user has rated
            sims = item_sim[i, rated_items]
            top_k = np.argsort(sims)[-k_neighbors:]
            top_sims = sims[top_k]
            top_ratings = user_ratings[rated_items[top_k]]
            
            denom = np.abs(top_sims).sum()
            if denom > 0:
                scores[i] = (top_sims * top_ratings).sum() / denom
        
        top_items = np.argsort(scores)[-n_recs:][::-1]
        recommendations[u + 1] = (top_items + 1).tolist()
    
    return recommendations

print("Computing item-based CF (this may take a minute)...")
top_k_item_cf = item_based_cf(R, k_neighbors=30, n_recs=10)

hr_icf = hit_rate_at_k(test, top_k_item_cf, k=10)
ndcg_icf = ndcg_at_k(test, top_k_item_cf, k=10)
print(f"\nItem-Based CF:")
print(f"  HR@10   = {hr_icf:.3f} ({hr_icf*100:.1f}%)")
print(f"  NDCG@10 = {ndcg_icf:.4f}")

In [ ]:
# ── Compare Methods ───────────────────────────────────────────────────────
results = pd.DataFrame({
    "Method": ["Popularity (Ch.1)", "User-Based CF", "Item-Based CF"],
    "HR@10": [0.42, hr_ucf, hr_icf],
    "NDCG@10": [0.0, ndcg_ucf, ndcg_icf]
})

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ["#95a5a6", "#3498db", "#e67e22"]
axes[0].bar(results["Method"], results["HR@10"] * 100, color=colors, edgecolor="white")
axes[0].axhline(85, color="red", linestyle="--", alpha=0.7, label="Target: 85%")
axes[0].set(ylabel="Hit Rate@10 (%)", title="Hit Rate@10 Comparison")
axes[0].legend()

axes[1].bar(results["Method"], results["NDCG@10"], color=colors, edgecolor="white")
axes[1].set(ylabel="NDCG@10", title="NDCG@10 Comparison")

plt.suptitle("Collaborative Filtering vs Popularity Baseline", y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig("img/cf_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(results.to_string(index=False))

In [ ]:
# ── Effect of k (Neighbors) on Item-Based CF ──────────────────────────────
k_values_nb = [5, 10, 20, 30, 50, 100]
hr_by_k = []

for k in k_values_nb:
    print(f"  Computing item-CF with k={k}...")
    recs = item_based_cf(R, k_neighbors=k, n_recs=10)
    hr_by_k.append(hit_rate_at_k(test, recs, k=10))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_values_nb, [h * 100 for h in hr_by_k], "o-", color="#e67e22", linewidth=2, markersize=8)
ax.set(xlabel="k (Number of Neighbors)", ylabel="Hit Rate@10 (%)",
       title="Item-Based CF: Effect of Neighbor Count")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("img/k_neighbors_effect.png", dpi=150, bbox_inches="tight")
plt.show()

for k, hr in zip(k_values_nb, hr_by_k):
    print(f"  k={k:>3d}: HR@10 = {hr:.3f} ({hr*100:.1f}%)")

In [ ]:
# ── Visualise Item Similarity Matrix (Top-20 Movies) ─────────────────────
item_sim_full = cosine_similarity(R.T)
np.fill_diagonal(item_sim_full, 0)

# Top 20 most-rated movies
top_20_ids = ratings.groupby("item_id").size().nlargest(20).index.values
top_20_idx = top_20_ids - 1

movies_df = pd.read_csv(
    url + "u.item", sep="|", encoding="latin-1", header=None,
    names=["item_id", "title"] + [f"c{i}" for i in range(22)],
    usecols=[0, 1]
)
titles = movies_df.set_index("item_id")["title"].to_dict()
labels = [titles.get(i, f"Movie {i}")[:25] for i in top_20_ids]

sim_subset = item_sim_full[np.ix_(top_20_idx, top_20_idx)]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(sim_subset, xticklabels=labels, yticklabels=labels,
            cmap="YlOrRd", vmin=0, vmax=1, ax=ax, square=True)
ax.set_title("Item-Item Cosine Similarity (Top-20 Most-Rated Movies)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("img/item_similarity_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## Progress Check

| # | Constraint | Target | Ch.2 Status |
|---|-----------|--------|-------------|
| 1 | ACCURACY | >85% HR@10 | ❌ ~68% (+26 from popularity!) |
| 2 | COLD START | New users/items | ❌ Fails for new users |
| 3 | SCALABILITY | 1M+ ratings | ⚠️ O(n²) similarity |
| 4 | DIVERSITY | Not just popular | ⚠️ Better — neighbors have diverse taste |
| 5 | EXPLAINABILITY | "Because you liked X" | ✅ "Users like you also watched..." |

**Bottom line**: 68% hit rate — personalisation jumps 26 points! But sparsity limits us. 93.7% of the matrix is empty.

**Next**: Ch.3 — Matrix Factorization → compress the sparse matrix into dense latent vectors.

## Exercises

**Exercise 1 — Pearson vs Cosine**  
Implement Pearson correlation for user-based CF (center each user's ratings by their mean before computing cosine similarity). Compare HR@10 against raw cosine.

**Exercise 2 — Minimum Overlap Threshold**  
Modify item-based CF to require at least `min_overlap=5` co-rated items before computing similarity. How does this affect HR@10?

**Exercise 3 — Hybrid: Popularity + CF**  
For users with fewer than 20 ratings, fall back to the popularity baseline. For users with ≥20 ratings, use item-based CF. Compare HR@10 against pure item-CF.

In [ ]:
# ── Exercise 1 scaffold — Pearson vs Cosine ───────────────────────────────
# TODO: Implement Pearson correlation for user-based CF
# Hint: Center each user's ratings by their mean before computing cosine

# R_centered = ... (subtract user means from non-zero entries)
# pearson_sim = cosine_similarity(R_centered)
# ... run user-based CF with pearson_sim, evaluate HR@10

pass

In [ ]:
# ── Exercise 2 scaffold — Minimum Overlap Threshold ──────────────────────
# TODO: Modify item similarity to require min_overlap co-rated items

# def item_sim_with_overlap(R_sparse, min_overlap=5):
#     """Set similarity to 0 for item pairs with fewer than min_overlap co-ratings."""
#     ...

pass

In [ ]:
# ── Exercise 3 scaffold — Hybrid Popularity + CF ─────────────────────────
# TODO: Use popularity for users with <20 ratings, item-CF for users with ≥20

# user_rating_counts = train.groupby("user_id").size()
# for user_id in test["user_id"].unique():
#     if user_rating_counts.get(user_id, 0) < 20:
#         recs = popularity_recs
#     else:
#         recs = item_cf_recs
# ... evaluate HR@10

pass